In [39]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
from transformers import AutoTokenizer


In [41]:
from datasets import load_dataset

# 1) Point to your actual CSV files
dataset_samsum = load_dataset(
    "csv",
    data_files={
        "train": "/content/drive/MyDrive/Colab_Notebooks/samsum/samsum-train.csv",
        "validation": "/content/drive/MyDrive/Colab_Notebooks/samsum/samsum-validation.csv",
        "test": "/content/drive/MyDrive/Colab_Notebooks/samsum/samsum-test.csv",
    }
)
print(dataset_samsum)
print(dataset_samsum["test"].column_names)


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})
['id', 'dialogue', 'summary']


In [42]:
sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]

In [43]:
sample_text

"Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye"

In [44]:
reference

"Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry."

In [45]:
print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [46]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

# Paths
tok_path = "/content/drive/MyDrive/Colab_Notebooks/tokenizer"
model_path = "/content/drive/MyDrive/Colab_Notebooks/pegasus-samsum-model"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(tok_path)

# Load model (GPU if available, else CPU with float32)
if torch.cuda.is_available():
    model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")
else:
    model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
        model_path, dtype=torch.float32
    ).to("cpu")

# Create summarization pipeline using your fine-tuned Pegasus model
pipe = pipeline(
    "summarization",
    model=model_pegasus,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Generation settings
gen_kwargs = {
    "min_length": 15,
    "max_length": 128,
    "num_beams": 8,
    "length_penalty": 0.8,
    "early_stopping": True,
    "no_repeat_ngram_size": 3
}

# Run on one sample from test set
sample_text = dataset_samsum["test"][0]["dialogue"]
summary = pipe(sample_text, **gen_kwargs)[0]["summary_text"]

# Display both dialogue and summary
print("📘 Dialogue:\n", sample_text[:400], "...")
print("\n🧾 Generated Summary:\n", summary)


[INFO|tokenization_utils_base.py:2093] 2025-11-07 10:37:05,706 >> loading file spiece.model
[INFO|tokenization_utils_base.py:2093] 2025-11-07 10:37:05,707 >> loading file tokenizer.json
[INFO|tokenization_utils_base.py:2093] 2025-11-07 10:37:05,708 >> loading file added_tokens.json
[INFO|tokenization_utils_base.py:2093] 2025-11-07 10:37:05,710 >> loading file special_tokens_map.json
[INFO|tokenization_utils_base.py:2093] 2025-11-07 10:37:05,711 >> loading file tokenizer_config.json
[INFO|tokenization_utils_base.py:2093] 2025-11-07 10:37:05,712 >> loading file chat_template.jinja
[INFO|configuration_utils.py:763] 2025-11-07 10:37:06,057 >> loading configuration file /content/drive/MyDrive/Colab_Notebooks/pegasus-samsum-model/config.json
[INFO|configuration_utils.py:839] 2025-11-07 10:37:06,060 >> Model config PegasusConfig {
  "activation_dropout": 0.1,
  "activation_function": "relu",
  "add_bias_logits": false,
  "add_final_layer_norm": true,
  "architectures": [
    "PegasusForCondit

📘 Dialogue:
 Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda:  ...

🧾 Generated Summary:
 Amanda can't find Betty's number. Larry called her last time they were at the park together. Hannah wants Amanda to text him.
